In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import joblib
import os

# Create directory for outputs
if not os.path.exists('processed_data'):
    os.makedirs('processed_data')

# Load the dataset
print("Loading data...")
df = pd.read_csv('kyrgyzstan_flood_data.csv')

# Convert date to datetime
df['date'] = pd.to_datetime(df['date'])

# Extract temporal features
print("Extracting temporal features...")
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['day_of_year'] = df['date'].dt.dayofyear
df['is_spring'] = ((df['month'] >= 3) & (df['month'] <= 5)).astype(int)
df['is_summer'] = ((df['month'] >= 6) & (df['month'] <= 8)).astype(int)

# Calculate days since last precipitation
print("Calculating advanced features...")
# Let's do this for each region separately
for region in df['region'].unique():
    region_data = df[df['region'] == region].sort_values('date')
    
    # Initialize with a high value (equivalent to "no recent precipitation")
    days_since_precip = np.ones(len(region_data)) * 30
    
    # Track days since precipitation for this region
    last_precip_day = None
    for i, (idx, row) in enumerate(region_data.iterrows()):
        if row['precipitation'] > 1.0:  # Significant precipitation (>1mm)
            last_precip_day = i
            days_since_precip[i] = 0
        elif last_precip_day is not None:
            days_since_precip[i] = i - last_precip_day
    
    # Update the dataframe
    df.loc[region_data.index, 'days_since_precip'] = days_since_precip

# Calculate cumulative precipitation (last 3 days, last 7 days, last 14 days)
print("Calculating cumulative precipitation...")
for region in df['region'].unique():
    region_data = df[df['region'] == region].sort_values('date')
    
    # Initialize arrays
    precip_3d = np.zeros(len(region_data))
    precip_7d = np.zeros(len(region_data))
    precip_14d = np.zeros(len(region_data))
    
    # Calculate rolling sums
    for i in range(len(region_data)):
        # Last 3 days
        if i >= 3:
            precip_3d[i] = region_data['precipitation'].iloc[i-3:i].sum()
        else:
            precip_3d[i] = region_data['precipitation'].iloc[:i+1].sum()
        
        # Last 7 days
        if i >= 7:
            precip_7d[i] = region_data['precipitation'].iloc[i-7:i].sum()
        else:
            precip_7d[i] = region_data['precipitation'].iloc[:i+1].sum()
        
        # Last 14 days
        if i >= 14:
            precip_14d[i] = region_data['precipitation'].iloc[i-14:i].sum()
        else:
            precip_14d[i] = region_data['precipitation'].iloc[:i+1].sum()
    
    # Update the dataframe
    df.loc[region_data.index, 'precip_3d'] = precip_3d
    df.loc[region_data.index, 'precip_7d'] = precip_7d
    df.loc[region_data.index, 'precip_14d'] = precip_14d

# Calculate rate of change in river level (from previous day)
print("Calculating river level dynamics...")
for region in df['region'].unique():
    region_data = df[df['region'] == region].sort_values('date')
    
    # Initialize with zeros
    river_level_change = np.zeros(len(region_data))
    
    # Calculate day-to-day change
    for i in range(1, len(region_data)):
        river_level_change[i] = region_data['river_level'].iloc[i] - region_data['river_level'].iloc[i-1]
    
    # Update the dataframe
    df.loc[region_data.index, 'river_level_change'] = river_level_change

# Calculate level-to-threshold ratio
df['level_to_threshold_ratio'] = df['river_level'] / df['flood_threshold']

# Create proximity to flooding feature
df['flood_proximity'] = (df['level_to_threshold_ratio'] > 0.8).astype(int)

# Create elevation ranges
df['elevation_range'] = pd.cut(df['elevation'], 
                               bins=[0, 1000, 2000, 4000], 
                               labels=['low', 'medium', 'high'])

# Handle missing values
print("Handling missing values...")
# For upstream_dam_level, fill with 0 if missing (indicating no upstream dam influence)
df['upstream_dam_level'].fillna(0, inplace=True)

# Check if there are any remaining missing values
missing_values = df.isnull().sum()
print("Remaining missing values after initial cleaning:")
print(missing_values[missing_values > 0])

# Drop any rows with missing values if they still exist
df.dropna(inplace=True)

# Prepare for ML: Split data by time for temporal validation
print("Splitting data for training and validation...")
# Let's use the last year for testing, the second-to-last year for validation
train_years = df['year'].min(), df['year'].max() - 2
val_year = df['year'].max() - 1
test_year = df['year'].max()

# Create train/validation/test sets
train_df = df[(df['year'] >= train_years[0]) & (df['year'] <= train_years[1])]
val_df = df[df['year'] == val_year]
test_df = df[df['year'] == test_year]

print(f"Train set: {len(train_df)} samples from years {train_years[0]}-{train_years[1]}")
print(f"Validation set: {len(val_df)} samples from year {val_year}")
print(f"Test set: {len(test_df)} samples from year {test_year}")

# Define features and target
target = 'flood_status'

# Features to use
numerical_features = [
    'temperature', 'precipitation', 'snowmelt', 'soil_moisture', 'river_level',
    'days_since_precip', 'precip_3d', 'precip_7d', 'precip_14d',
    'river_level_change', 'level_to_threshold_ratio'
]

categorical_features = [
    'region', 'basin', 'elevation_range', 'month'
]

# Remove the target from features
features = numerical_features + categorical_features

# Create X and y for each set
X_train = train_df[features]
y_train = train_df[target]

X_val = val_df[features]
y_val = val_df[target]

X_test = test_df[features]
y_test = test_df[target]

# Create preprocessing pipeline
print("Creating preprocessing pipeline...")
# Numeric features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical features
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Fit the preprocessor to the training data
print("Fitting preprocessing pipeline...")
preprocessor.fit(X_train)

# Transform the data
print("Transforming data...")
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# Save the preprocessed data
print("Saving preprocessed data...")
joblib.dump(preprocessor, 'processed_data/preprocessor.pkl')
np.save('processed_data/X_train.npy', X_train_processed)
np.save('processed_data/y_train.npy', y_train.values)
np.save('processed_data/X_val.npy', X_val_processed)
np.save('processed_data/y_val.npy', y_val.values)
np.save('processed_data/X_test.npy', X_test_processed)
np.save('processed_data/y_test.npy', y_test.values)

# Save feature names and mapping for interpretability
feature_names = numerical_features.copy()

# For categorical features, we need to get the encoded feature names
cat_feature_names = []
for i, feature in enumerate(categorical_features):
    encoder = preprocessor.transformers_[1][1].steps[1][1]  # Get the OneHotEncoder
    categories = encoder.categories_[i]
    for category in categories:
        cat_feature_names.append(f"{feature}_{category}")

all_feature_names = feature_names + cat_feature_names
with open('processed_data/feature_names.txt', 'w') as f:
    for feature in all_feature_names:
        f.write(f"{feature}\n")

# Save processed dataframes for reference
train_df.to_csv('processed_data/train_data.csv', index=False)
val_df.to_csv('processed_data/validation_data.csv', index=False)
test_df.to_csv('processed_data/test_data.csv', index=False)

print("Preprocessing complete! Processed data saved to 'processed_data' directory.")

Loading data...
Extracting temporal features...
Calculating advanced features...
Calculating cumulative precipitation...
Calculating river level dynamics...
Handling missing values...
Remaining missing values after initial cleaning:
Series([], dtype: int64)
Splitting data for training and validation...
Train set: 26298 samples from years 2018-2021
Validation set: 6570 samples from year 2022
Test set: 6570 samples from year 2023
Creating preprocessing pipeline...
Fitting preprocessing pipeline...
Transforming data...
Saving preprocessed data...


C:\Users\Aelina.Daniiarkyzy\AppData\Local\Temp\ipykernel_7128\133361288.py:116: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['upstream_dam_level'].fillna(0, inplace=True)


Preprocessing complete! Processed data saved to 'processed_data' directory.
